# USLegalQA - F1: QLoRA fine-tuning

Trains a QLoRA adapter on the opinion-level training split and generates
predictions on the test split for direct comparison against the baselines.

## Why this notebook is defensive

The earlier version of this work (v2) contained training bugs that silently
produced a broken model while reporting plausible numbers. Each is checked
**before** training starts, and the notebook refuses to proceed if a check fails.

| v2 bug | Consequence | Check |
|---|---|---|
| `</s>` appended as EOS | Llama-3 has no such token; it tokenised as ordinary text, so no EOS was learned and generation ran to `max_new_tokens`, collapsing ROUGE precision | 4.1 |
| `pad_token = eos_token` | The collator masks every token equal to `pad_token_id` to -100, so even a correct EOS is excluded from the loss | 4.2 |
| Loss over the whole sequence | Most gradient spent reproducing identical instruction boilerplate | 4.3 |
| Train/inference template mismatch | Model queried off-distribution at evaluation | 4.4 |
| Silent truncation | Long examples trained on fragments | 4.5 |
| `temperature=0.1` with `do_sample=False` | Silently ignored; reporting it would mislead | 6 |
| `repetition_penalty=1.3` | Penalises the repetition legal text requires | 6 |
| Early stopping claimed, never implemented | Reported method did not match the code | 5 |

## Design decision: F1 is trained open-book

B2 supplies the opinion passage and scores 0.3709 rescaled BERTScore with no
training at all. Training F1 closed-book would compare a trained model without
the passage against an untrained model with it, so the passage rather than the
training would explain the difference.

F1 therefore receives the same passage B2 receives. The comparison then isolates
what fine-tuning contributes given the same information, which is what RQ3 asks.

---

### Before running
1. Accelerator -> **GPU T4 x2**
2. Internet -> **On**
3. Add Data -> a dataset with `train_with_passages.jsonl` and `test_with_passages.jsonl`
4. Add-ons -> Secrets -> `HF_TOKEN` (ticked)

Prepare both files locally first:
```powershell
python -m uslegalqa.prepare_kaggle --test data/dataset/splits/train.jsonl --outdir data/kaggle_train
python -m uslegalqa.prepare_kaggle --test data/dataset/splits/test.jsonl  --outdir data/kaggle_test
```
Rename the first to `train_with_passages.jsonl` before uploading.

In [1]:
# --- configuration ---

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

SEED           = 2        # rerun with 1 and 2 for the three-seed report
OPEN_BOOK      = True      # see the design note above
MAX_SEQ_LENGTH = 1024
EPOCHS         = 1
LR             = 2e-4
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TRAIN_BATCH    = 4
GRAD_ACCUM     = 4         
EVAL_BATCH     = 4
MAX_NEW_TOKENS = 160
TRAIN_LIMIT    = None      # e.g. 2000 for a quick pipeline test
EVAL_LIMIT     = None

INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
RUN_NAME   = f"f1_seed{SEED}" + ("" if OPEN_BOOK else "_closedbook")

print(f"model     {MODEL_ID}")
print(f"run       {RUN_NAME}")
print(f"open-book {OPEN_BOOK}")
print(f"seed      {SEED}, epochs {EPOCHS}, lr {LR}, LoRA r={LORA_R}")
print(f"effective batch {TRAIN_BATCH * GRAD_ACCUM}")

model     meta-llama/Llama-3.2-3B-Instruct
run       f1_seed1
open-book True
seed      1, epochs 1, lr 0.0002, LoRA r=16
effective batch 16


## 1. Environment

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "peft", "trl", "accelerate", "bitsandbytes",
                "datasets"], check=False)

import torch, transformers, peft, random, numpy as np
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)

# Seed everything: a single-seed result reported as a point estimate is not
# defensible, and reproducibility requires the seed to actually bind.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

assert torch.cuda.is_available(), "No GPU -- set Accelerator to GPU T4 x2"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.2 MB/s eta 0:00:00
torch        2.10.0+cu128
transformers 5.17.0
peft         0.21.0
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


## 2. Data

In [3]:
import json, os, glob

def find_input(filename):
    hits = glob.glob(f"{INPUT_DIR}/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {INPUT_DIR}")
    return hits[0]

def load(filename, limit=None):
    recs = [json.loads(l) for l in open(find_input(filename), encoding="utf-8")
            if l.strip()]
    return recs[:limit] if limit else recs

train = load("train_with_passages.jsonl", TRAIN_LIMIT)
test  = load("test_with_passages.jsonl", EVAL_LIMIT)

print(f"train {len(train)} pairs / {len({r['cluster_id'] for r in train})} opinions")
print(f"test  {len(test)} pairs / {len({r['cluster_id'] for r in test})} opinions")

# The split must be by opinion. With ~16 pairs per opinion, any overlap means
# the model is evaluated on documents it was trained on.
overlap = {r["cluster_id"] for r in train} & {r["cluster_id"] for r in test}
assert not overlap, (
    f"LEAKAGE: {len(overlap)} opinions appear in both splits. "
    "Regenerate with `python -m uslegalqa.split`.")
print("\nNo opinion appears in both splits.")

train 14308 pairs / 878 opinions
test  1771 pairs / 109 opinions

No opinion appears in both splits.


## 3. Prompt construction

One function builds both the training text and the inference prompt, so they
cannot drift apart. In v2 the training text ended `### Answer:\n{answer}` while
the inference prompt ended `### Answer:` with no newline - a small difference
that put the model off-distribution on every evaluation item.

In [4]:
INSTRUCTION_OPEN = """You are a US legal expert. Answer the question using only the passage from the Supreme Court opinion below.

CASE: {case_name} ({date_filed})

PASSAGE:
{passage}

QUESTION: {question}"""

INSTRUCTION_CLOSED = """You are a US legal expert. Answer the question about the Supreme Court case below.

CASE: {case_name} ({date_filed})

QUESTION: {question}"""

RESPONSE_MARKER = "\n\nANSWER: "


def build_prompt(rec):
    """The prompt half -- identical at training and inference."""
    template = INSTRUCTION_OPEN if OPEN_BOOK else INSTRUCTION_CLOSED
    body = template.format(case_name=rec["case_name"],
                           date_filed=rec.get("date_filed", ""),
                           passage=rec.get("passage", ""),
                           question=rec["question"])
    return body + RESPONSE_MARKER


def build_training_text(rec, eos):
    """Prompt plus reference answer plus an explicit EOS.

    The EOS is what teaches the model to stop. Omitting it, or appending a
    token the tokenizer does not recognise, produces a model that generates
    until it hits max_new_tokens.
    """
    return build_prompt(rec) + rec["answer"].strip() + eos


print(build_prompt(train[0])[:700])
print("...\n[answer follows here, then EOS]")

You are a US legal expert. Answer the question using only the passage from the Supreme Court opinion below.

CASE: Whitfield v. United States (2005-01-11)

PASSAGE:
 delivered the opinion of the Court.
These cases present the question whether conviction for conspiracy to commit money laundering, in violation of 18 U. S. C. § 1956(h), requires proof of an overt act in furtherance of the conspiracy. We hold that it does not.

I
In March 1999, a federal grand jury returned a 20-count indictment against petitioners and five codefendants. As relevant here, Count II of the indictment charged petitioners with conspiracy to launder money, in violation of § 1956(h). The indictment described, in gener
...
[answer follows here, then EOS]


## 4. Tokenizer and the v2 precondition checks

In [5]:
from transformers import AutoTokenizer

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret -- fine for ungated models")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
print(f"\neos_token {tokenizer.eos_token!r} id={tokenizer.eos_token_id}")
print(f"pad_token {tokenizer.pad_token!r} id={tokenizer.pad_token_id}")

HF_TOKEN loaded


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]


eos_token '<|eot_id|>' id=128009
pad_token None id=None


### 4.1 The EOS token must be the model's own

v2 appended the literal `</s>`, which is Llama-2's EOS. The Llama-3 tokenizer
has no such token, so it encoded as three ordinary text tokens. No EOS ever
entered the labels, the model never learned to stop, and generation ran to
`max_new_tokens` - which is why ROUGE precision collapsed while BERTScore
stayed high.

In [6]:
EOS = tokenizer.eos_token
assert EOS is not None, "tokenizer has no eos_token"

# The v2 failure, made explicit.
wrong_ids = tokenizer("</s>", add_special_tokens=False)["input_ids"]
print(f"'</s>' encodes to {wrong_ids} -> "
      f"{tokenizer.convert_ids_to_tokens(wrong_ids)}")
if len(wrong_ids) > 1:
    print("  CONFIRMED: '</s>' is NOT a single token for this tokenizer.")
    print("  v2 appended it as text, so no EOS was ever learned.")

eos_ids = tokenizer(EOS, add_special_tokens=False)["input_ids"]
assert len(eos_ids) == 1, f"{EOS!r} is not a single token: {eos_ids}"
print(f"\n{EOS!r} encodes to the single token {eos_ids[0]} -- correct.")

'</s>' encodes to [524, 82, 29] -> ['</', 's', '>']
  CONFIRMED: '</s>' is NOT a single token for this tokenizer.
  v2 appended it as text, so no EOS was ever learned.

'<|eot_id|>' encodes to the single token 128009 -- correct.


### 4.2 The pad token must differ from EOS

The collator masks every token equal to `pad_token_id` to -100. If padding and
EOS share an id, even a correct EOS is excluded from the loss and the model
still never learns to stop. v2 set `pad_token = eos_token`, so both bugs were
present at once.

In [7]:
PAD_CANDIDATES = ["<|finetune_right_pad_id|>", "<|reserved_special_token_0|>"]

if tokenizer.pad_token_id is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    chosen = None
    for cand in PAD_CANDIDATES:
        cid = tokenizer.convert_tokens_to_ids(cand)
        if cid is not None and cid >= 0 and cid != tokenizer.unk_token_id:
            tokenizer.pad_token = cand
            chosen = cand
            break
    if chosen is None:
        tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
        chosen = "<|pad|>"
        print("Added a new pad token -- embeddings will be resized.")
    print(f"pad_token set to {chosen!r} (id {tokenizer.pad_token_id})")

assert tokenizer.pad_token_id != tokenizer.eos_token_id, (
    "pad_token_id == eos_token_id: the collator would mask every EOS out of "
    "the loss and the model would never learn to stop.")
print(f"\npad {tokenizer.pad_token_id} != eos {tokenizer.eos_token_id} -- correct.")

tokenizer.padding_side = "right"   # right for training
print("padding_side = 'right' for training (switched to 'left' before generation)")

pad_token set to '<|finetune_right_pad_id|>' (id 128004)

pad 128004 != eos 128009 -- correct.
padding_side = 'right' for training (switched to 'left' before generation)


### 4.3 Loss must cover the answer only

v2 computed loss over the entire sequence, so most of the gradient went into
reproducing instruction boilerplate identical across all 6,664 examples. Here
the prompt tokens are masked to -100 and only the answer and its EOS
contribute.

In [8]:
import torch

def tokenize_example(rec):
    """Tokenize one example with completion-only loss masking.

    Two constraints have to hold simultaneously:

    1. The EOS must be the final token. It is appended as a token ID rather
       than as text, because a tokenizer can merge a string EOS into the
       preceding characters, leaving the sequence without a clean stopping
       point -- the v2 failure in a subtler form.

    2. The answer must survive. It is the only part contributing to the loss,
       so an example whose answer is truncated away trains on nothing.

    The passage is the only variable-length component, so when a sequence does
    not fit, the PASSAGE is shortened rather than the sequence as a whole.
    Truncating the sequence would drop the answer from the end; truncating the
    passage preserves the supervision signal at the cost of some context.
    """
    answer_ids = tokenizer(rec["answer"].strip(),
                           add_special_tokens=False)["input_ids"]
    # Reserve room for the answer and its EOS.
    reserve = len(answer_ids) + 1

    passage = rec.get("passage", "")
    for _ in range(8):
        shrunk = dict(rec)
        shrunk["passage"] = passage
        prompt_ids = tokenizer(build_prompt(shrunk),
                               add_special_tokens=True)["input_ids"]
        if len(prompt_ids) + reserve <= MAX_SEQ_LENGTH:
            break
        overflow = len(prompt_ids) + reserve - MAX_SEQ_LENGTH
        # ~4 characters per token, with a margin so the loop converges.
        cut = max(200, int(overflow * 4 * 1.2))
        if len(passage) <= cut:
            passage = passage[:max(0, len(passage) // 2)]
        else:
            # Trim from the end: the supporting span sits near the middle, and
            # the opening of a passage carries the procedural setup.
            passage = passage[:len(passage) - cut]
        if not passage:
            break
    else:
        prompt_ids = tokenizer(build_prompt({**rec, "passage": passage}),
                               add_special_tokens=True)["input_ids"]

    # Last resort: if even an empty passage does not fit, clip the answer
    # rather than emit an example that trains on nothing.
    if len(prompt_ids) + reserve > MAX_SEQ_LENGTH:
        prompt_ids = prompt_ids[:MAX_SEQ_LENGTH - 2]
        answer_ids = answer_ids[:1]

    input_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
    labels = ([-100] * len(prompt_ids)) + answer_ids + [tokenizer.eos_token_id]

    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}


ex = tokenize_example(train[0])
n_total = len(ex["labels"])
n_masked = sum(1 for l in ex["labels"] if l == -100)
print(f"tokens {n_total}: {n_masked} masked (prompt), "
      f"{n_total-n_masked} trained (answer)")

assert n_total - n_masked > 0, "every token masked -- nothing would be learned"
assert n_masked > 0, "no tokens masked -- loss covers the prompt (v2 defect)"
assert ex["labels"][-1] == tokenizer.eos_token_id, (
    f"last label is {ex['labels'][-1]}, expected eos {tokenizer.eos_token_id}. "
    "The model would not learn to stop.")
print(f"final label is EOS ({tokenizer.eos_token_id}), unmasked -- correct.")
print("\ntokens contributing to loss:")
print("  " + tokenizer.decode([l for l in ex["labels"] if l != -100])[:220])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


tokens 763: 680 masked (prompt), 83 trained (answer)
final label is EOS (128009), unmasked -- correct.

tokens contributing to loss:
  The defendants operated a deceptive investment program that promised investors double their money within eighteen months through purported overseas investments in mining and commodities. The scheme collected over $400 mi


### 4.4 Training and inference prompts must be byte-identical

In [9]:
train_text = build_training_text(train[0], EOS)
infer_prompt = build_prompt(train[0])

assert train_text.startswith(infer_prompt), (
    "training text does not begin with the inference prompt -- the model "
    "would be queried off-distribution at evaluation")
print("training text starts with the exact inference prompt -- correct.")
print(f"\nprompt ends:        ...{infer_prompt[-60:]!r}")
print(f"training continues: {train_text[len(infer_prompt):][:80]!r}")

training text starts with the exact inference prompt -- correct.

prompt ends:        ...'nal Church, and what financial scale did it reach?\n\nANSWER: '
training continues: 'The defendants operated a deceptive investment program that promised investors d'


### 4.5 The answer must survive truncation

The answer is the only part of the sequence contributing to the loss, so an
example whose answer is truncated away trains on nothing.

The passage is the only variable-length component of the prompt, and a few
passages are far larger than typical -- `passage_for` takes a fixed window
either side of the supporting span, and some spans are several hundred
characters. When a sequence does not fit, the **passage** is shortened rather
than the sequence as a whole: truncating the sequence drops the answer from the
end, whereas shortening the passage costs some context but preserves the
supervision signal.

This cell reports how often that happens, so the trade-off can be stated in the
write-up rather than left implicit.

In [10]:
lens, shortened, answer_lost, answer_clipped = [], 0, 0, 0
for rec in train:
    a = tokenizer(rec["answer"].strip(), add_special_tokens=False)["input_ids"]
    p = tokenizer(build_prompt(rec), add_special_tokens=True)["input_ids"]
    lens.append(len(p) + len(a) + 1)

    ex_t = tokenize_example(rec)
    trained = [l for l in ex_t["labels"] if l != -100]
    if len(trained) <= 1:
        answer_lost += 1
    elif len(trained) < len(a) + 1:
        answer_clipped += 1
    if len(ex_t["input_ids"]) < len(p) + len(a) + 1:
        shortened += 1

lens.sort()
p99 = lens[int(len(lens) * 0.99)]
print(f"untruncated sequence tokens: min {lens[0]} / "
      f"median {lens[len(lens)//2]} / p95 {lens[int(len(lens)*0.95)]} / "
      f"p99 {p99} / max {lens[-1]}")
print(f"MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}")
print(f"passage shortened to fit: {shortened}/{len(train)} "
      f"({100*shortened/len(train):.1f}%)")
print(f"answer entirely lost:     {answer_lost}")
print(f"answer partially clipped: {answer_clipped}")

# The answer is the only supervision signal, so losing it is fatal. Shortening
# the passage is acceptable and is reported so it can be stated in the write-up.
assert answer_lost == 0, (
    f"{answer_lost} examples still train on nothing. This should not happen "
    "now that the passage is shortened first -- inspect those records.")
assert answer_clipped < len(train) * 0.01, (
    f"{answer_clipped} answers clipped ({100*answer_clipped/len(train):.1f}%). "
    f"Raise MAX_SEQ_LENGTH to {p99}.")

if shortened > len(train) * 0.05:
    print(f"\nNote: {100*shortened/len(train):.1f}% of examples had their "
          f"passage shortened. Raising MAX_SEQ_LENGTH to {p99} would avoid "
          "this and preserve full context.")
else:
    print("\nAll answers preserved; passage shortening is rare.")


untruncated sequence tokens: min 281 / median 672 / p95 879 / p99 1184 / max 2003
MAX_SEQ_LENGTH = 1024
passage shortened to fit: 285/14308 (2.0%)
answer entirely lost:     0
answer partially clipped: 0

All answers preserved; passage shortening is rare.


## 5. Train

In [11]:
from datasets import Dataset
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import random as _r

# Inner validation for early stopping, carved from TRAIN and split by opinion
# so the stopping signal is not contaminated by the test set.
train_ops = sorted({r["cluster_id"] for r in train})
_r.Random(SEED).shuffle(train_ops)
val_ops = set(train_ops[:max(1, int(0.05*len(train_ops)))])

train_split = [r for r in train if r["cluster_id"] not in val_ops]
val_split   = [r for r in train if r["cluster_id"] in val_ops]
print(f"train {len(train_split)} / inner-val {len(val_split)} "
      f"({len(val_ops)} opinions held out)")

ds_train = Dataset.from_list([tokenize_example(r) for r in train_split])
ds_val   = Dataset.from_list([tokenize_example(r) for r in val_split])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", token=hf_token)

if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))
    print("resized token embeddings for the added pad token")

model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False          # required with gradient checkpointing

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

train 13659 / inner-val 649 (43 opinions held out)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [12]:
# DataCollatorForSeq2Seq pads labels with -100, preserving the masking.
# DataCollatorForLanguageModeling would rebuild labels from input_ids and
# discard it -- which is how v2 lost both the masking and the EOS.
collator = DataCollatorForSeq2Seq(tokenizer, padding=True,
                                  label_pad_token_id=-100)

# TrainingArguments changed between transformers 4.x and 5.x: some arguments
# were renamed and others removed. Rather than pin a version, the accepted
# parameter names are read from the signature and the configuration is filtered
# to match, with known renames applied. Anything dropped is reported, so a
# silently ignored setting cannot end up misdescribed in the methods section.
import inspect

accepted = set(inspect.signature(TrainingArguments.__init__).parameters)

wanted = {
    "output_dir": f"{OUTPUT_DIR}/{RUN_NAME}",
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "per_device_eval_batch_size": TRAIN_BATCH,
    "learning_rate": LR,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 25,
    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": True,
    "optim": "paged_adamw_8bit",
    "gradient_checkpointing": True,
    "report_to": "none",
    "seed": SEED,
    "dataloader_num_workers": 2,
}

# Alternative spellings across versions, tried in order.
ALIASES = {
    "eval_strategy": ["evaluation_strategy"],
    "warmup_ratio": ["warmup_steps_ratio", "lr_warmup_ratio"],
    "fp16": ["use_fp16"],
}

final, dropped, renamed = {}, [], []
for key, value in wanted.items():
    if key in accepted:
        final[key] = value
        continue
    placed = False
    for alt in ALIASES.get(key, []):
        if alt in accepted:
            final[alt] = value
            renamed.append(f"{key} -> {alt}")
            placed = True
            break
    if not placed:
        dropped.append(key)

if renamed:
    print("renamed for this transformers version:")
    for r in renamed:
        print(f"  {r}")
if dropped:
    print("NOT SUPPORTED by this transformers version and therefore unset:")
    for d in dropped:
        print(f"  {d} (wanted {wanted[d]!r})")
    print("  -> do not describe these in the methods section")

args = TrainingArguments(**final)

# warmup_ratio has no universal equivalent; if it was dropped, approximate it
# with an explicit step count so the schedule still has a warmup phase.
if "warmup_ratio" in dropped and "warmup_steps" in accepted:
    steps_per_epoch = max(1, len(ds_train) // (TRAIN_BATCH * GRAD_ACCUM))
    total = steps_per_epoch * EPOCHS
    args.warmup_steps = max(1, int(0.03 * total))
    print(f"\nwarmup_steps set to {args.warmup_steps} "
          f"(~3% of {total} total steps)")

print(f"\neffective batch {TRAIN_BATCH * GRAD_ACCUM}, "
      f"{len(final)} arguments applied")

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val, data_collator=collator,
    # v2 described early stopping but registered no callback, so training ran
    # the full schedule. Registered here so the method matches the code.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

# Inspect one collated batch before committing to a long run.
batch = collator([ds_train[i] for i in range(min(2, len(ds_train)))])
assert (batch["labels"] == -100).any(), "collator discarded the label masking"
assert (batch["labels"] == tokenizer.eos_token_id).any(), \
    "no EOS survived collation -- the model will not learn to stop"
print("collated batch preserves masking and EOS -- correct.\n")

result = trainer.train()
print(f"\ntrain loss {result.training_loss:.4f}")

adapter_dir = f"{OUTPUT_DIR}/{RUN_NAME}_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"adapter saved to {adapter_dir}")


NOT SUPPORTED by this transformers version and therefore unset:
  warmup_ratio (wanted 0.03)
  -> do not describe these in the methods section

warmup_steps set to 25 (~3% of 853 total steps)

effective batch 16, 22 arguments applied
collated batch preserves masking and EOS -- correct.



Step,Training Loss,Validation Loss
100,1.139295,1.137161
200,1.107896,1.112042
300,1.106974,1.093726
400,1.074797,1.073148
500,1.078234,1.061674
600,1.066461,1.051335
700,1.053698,1.045348
800,1.020635,1.043039
854,1.017045,1.042972


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6aafc251-260b9d764d2ea2815caf375c;5d206a24-c346-4b53-a660-63615f4afa3e)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root


train loss 1.0920


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6ab00c13-3c080b872fd426cb1ff2d1c4;58ea2ef5-8b96-4df8-9306-1c4d04eef937)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


adapter saved to /kaggle/working/f1_seed1_adapter


## 6. Generate predictions

Greedy decoding, no repetition penalty.

v2 set `temperature=0.1` alongside `do_sample=False`, where temperature is
silently ignored - reporting it in a methods section would have been
misleading. It also set `repetition_penalty=1.3`, well above the usual
1.0-1.15, which penalises exactly the repetition legal prose requires: party
names, statutory citations, "the Court".

In [13]:
import time, gc

model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"     # left padding for batched generation


def generate(prompts, size):
    """Greedy batched generation with back-off on out-of-memory."""
    while size >= 1:
        try:
            out_all = []
            for i in range(0, len(prompts), size):
                enc = tokenizer(prompts[i:i+size], return_tensors="pt",
                                padding=True, truncation=True,
                                max_length=MAX_SEQ_LENGTH).to(model.device)
                with torch.no_grad():
                    out = model.generate(
                        **enc, max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,                 # greedy
                        repetition_penalty=1.0,          # see note above
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id)
                # Slice off the prompt: scoring the full sequence would
                # inflate overlap with anything prompt-adjacent.
                gen = out[:, enc["input_ids"].shape[-1]:]
                out_all += [t.strip() for t in
                            tokenizer.batch_decode(gen, skip_special_tokens=True)]
                del enc, out, gen
            return out_all
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            size //= 2
            print(f"    out of memory -- retrying at batch size {size}", flush=True)
    raise RuntimeError("out of memory even at batch size 1")


out_path = f"{OUTPUT_DIR}/{RUN_NAME}.jsonl"
done = set()
if os.path.exists(out_path):
    for line in open(out_path, encoding="utf-8"):
        if line.strip():
            try:
                d = json.loads(line)
                done.add((d["cluster_id"], d["question"]))
            except Exception:
                pass
    print(f"resuming: {len(done)} already written")

todo = [r for r in test if (r["cluster_id"], r["question"]) not in done]
print(f"{len(todo)} predictions to generate\n")

start, empty = time.time(), 0
with open(out_path, "a", encoding="utf-8") as f:
    for b in range(0, len(todo), EVAL_BATCH):
        batch = todo[b:b+EVAL_BATCH]
        for rec, pred in zip(batch, generate([build_prompt(r) for r in batch],
                                             EVAL_BATCH)):
            if not pred.strip():
                empty += 1
            f.write(json.dumps({
                "cluster_id": rec["cluster_id"],
                "question": rec["question"],
                "answer": rec["answer"],
                "prediction": pred,
                "question_type": rec.get("question_type"),
                "category": rec.get("category"),
                "system": RUN_NAME,
            }, ensure_ascii=False) + "\n")
        if (b // EVAL_BATCH) % 20 == 0:
            f.flush()
            n = b + len(batch)
            rate = n / max(time.time()-start, 1e-9)
            print(f"  [{n}/{len(todo)}] {rate:.1f} items/s, "
                  f"~{(len(todo)-n)/max(rate,1e-9)/60:.0f} min left, "
                  f"{empty} empty", flush=True)

print(f"\ndone in {(time.time()-start)/60:.1f} min, {empty} empty -> {out_path}")

1771 predictions to generate

  [4/1771] 0.4 items/s, ~77 min left, 0 empty
  [84/1771] 0.3 items/s, ~88 min left, 0 empty
  [164/1771] 0.3 items/s, ~81 min left, 0 empty
  [244/1771] 0.3 items/s, ~77 min left, 0 empty
  [324/1771] 0.3 items/s, ~73 min left, 0 empty
  [404/1771] 0.3 items/s, ~70 min left, 0 empty
  [484/1771] 0.3 items/s, ~66 min left, 0 empty
  [564/1771] 0.3 items/s, ~63 min left, 0 empty
  [644/1771] 0.3 items/s, ~58 min left, 0 empty
  [724/1771] 0.3 items/s, ~54 min left, 0 empty
  [804/1771] 0.3 items/s, ~49 min left, 0 empty
  [884/1771] 0.3 items/s, ~45 min left, 0 empty
  [964/1771] 0.3 items/s, ~41 min left, 0 empty
  [1044/1771] 0.3 items/s, ~37 min left, 0 empty
  [1124/1771] 0.3 items/s, ~33 min left, 0 empty
  [1204/1771] 0.3 items/s, ~29 min left, 0 empty
  [1284/1771] 0.3 items/s, ~25 min left, 0 empty
  [1364/1771] 0.3 items/s, ~21 min left, 0 empty
  [1444/1771] 0.3 items/s, ~17 min left, 0 empty
  [1524/1771] 0.3 items/s, ~13 min left, 0 empty
  [160

## 7. Did the EOS fix work?

The decisive check. v2's model generated to `max_new_tokens` because it never
learned to stop, giving a length ratio above 2 and collapsed ROUGE precision.
A correctly trained model produces answers close to reference length.

In [14]:
recs = [json.loads(l) for l in open(out_path, encoding="utf-8") if l.strip()]
pl = [len(r["prediction"].split()) for r in recs]
rl = [len(r["answer"].split()) for r in recs]
ratio = (sum(pl)/len(pl)) / max(sum(rl)/len(rl), 1e-9)
dupes = len(recs) - len({r["prediction"] for r in recs})
empty = sum(1 for x in pl if x == 0)
capped = sum(1 for x in pl if x > MAX_NEW_TOKENS*0.9)

print(f"{RUN_NAME}: {len(recs)} predictions")
print(f"  prediction words  {sum(pl)/len(pl):.1f}")
print(f"  reference words   {sum(rl)/len(rl):.1f}")
print(f"  length ratio      {ratio:.2f}")
print(f"  empty             {empty}")
print(f"  duplicates        {dupes} ({100*dupes/len(recs):.1f}%)")
print(f"  near token cap    {capped}")
print()
if ratio > 2.0:
    print("  FAIL: generation is running long. The model did not learn to stop")
    print("  -- recheck sections 4.1 to 4.3 before trusting any score.")
elif ratio > 1.4:
    print("  Somewhat verbose but bounded. ROUGE precision will be suppressed;")
    print("  report precision and recall separately.")
else:
    print("  Length well calibrated: the EOS token was learned.")

r = recs[0]
print(f"\n  Q:          {r['question'][:100]}")
print(f"  reference:  {r['answer'][:160]}")
print(f"  prediction: {r['prediction'][:160]}")

f1_seed1: 1771 predictions
  prediction words  60.3
  reference words   58.6
  length ratio      1.03
  empty             0
  duplicates        0 (0.0%)
  near token cap    0

  Length well calibrated: the EOS token was learned.

  Q:          What constitutional violations did Dotson claim resulted from Ohio's parole procedures?
  reference:  Dotson contended that applying parole guidelines adopted in 1998 retroactively to his case, which predated those guidelines, violated two constitutional protect
  prediction: Dotson alleged that the state violated the Ex Post Facto Clause and Due Process Clause by retroactively applying new, harsher parole guidelines to his case, whi


## 8. Download and score

Download `f1_seed42.jsonl` from **Output**, put it in `data/predictions/`, then:

```powershell
python -m uslegalqa.evaluate `
  --preds data/predictions/b0.jsonl `
  --preds data/predictions/b1.jsonl `
  --preds data/predictions/b2.jsonl `
  --preds data/predictions/f1_seed42.jsonl `
  --reference b2 --breakdowns
```

`--reference b2` is deliberate. B2 is the strongest baseline at 0.3709 rescaled
BERTScore with no training at all, so the question RQ3 actually asks is whether
fine-tuning beats prompting given the same passage.

### Then two more seeds

Set `SEED` to 1, then 2, and rerun. Report mean and standard deviation across
the three.

### If F1 does not beat B2

That is a finding, not a failure: for this task at this model scale, supplying
the passage matters more than adapting the weights. Report it plainly - an
honest negative result on RQ3 is worth more than a tuned number.

In [15]:
print("Ready for download:\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f"{OUTPUT_DIR}/{f}"
    if f.endswith(".jsonl"):
        n = sum(1 for _ in open(p, encoding="utf-8"))
        print(f"  {f:<30} {n:>6} records")
    elif os.path.isdir(p) and "adapter" in f:
        mb = sum(os.path.getsize(os.path.join(p,x)) for x in os.listdir(p))/1e6
        print(f"  {f+'/':<30} {mb:>6.1f} MB  (LoRA adapter)")

Ready for download:

  f1_seed1.jsonl                   1771 records
  f1_seed1_adapter/               114.5 MB  (LoRA adapter)
